<a href="https://colab.research.google.com/github/rajilsaj/nasa-mosaics-project/blob/xgboost/notebooks/05_train_xgboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# XGBOOST TRAINING - MARS VORTEX DETECTION
# ==========================================

import pandas as pd
import numpy as np
import xgboost as xgb
import os
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from sklearn.metrics import roc_curve, auc

drive.mount('/content/drive')

WINDOW_DIR = "/content/drive/MyDrive/2026/www/nasa-mosaics-project/data/windows"
MODEL_DIR = "/content/drive/MyDrive/2026/www/nasa-mosaics-project/models"

os.makedirs(MODEL_DIR, exist_ok=True)

print("Model directory:", MODEL_DIR)



Mounted at /content/drive
Model directory: /content/drive/MyDrive/2026/www/nasa-mosaics-project/models


In [2]:
import shutil
import os

# Create folder if it doesn't exist
os.makedirs(MODEL_DIR, exist_ok=True)

# Delete everything inside the folder
for filename in os.listdir(MODEL_DIR):
    file_path = os.path.join(MODEL_DIR, filename)

    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.remove(file_path)
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)
        print("Deleted:", file_path)
    except Exception as e:
        print("Failed to delete %s. Reason: %s" % (file_path, e))

print("MODEL_DIR cleaned.")


Deleted: /content/drive/MyDrive/2026/www/nasa-mosaics-project/models/xgb_vortex_model.json
Deleted: /content/drive/MyDrive/2026/www/nasa-mosaics-project/models/confusion_matrix.png
Deleted: /content/drive/MyDrive/2026/www/nasa-mosaics-project/models/roc_curve.png
Deleted: /content/drive/MyDrive/2026/www/nasa-mosaics-project/models/precision_recall_curve.png
Deleted: /content/drive/MyDrive/2026/www/nasa-mosaics-project/models/metrics_vs_threshold.png
MODEL_DIR cleaned.


In [3]:
train_df = pd.read_csv(f"{WINDOW_DIR}/train_features.csv")
val_df = pd.read_csv(f"{WINDOW_DIR}/val_features.csv")

print("Train:", train_df.shape)
print("Val:", val_df.shape)


Train: (92, 8)
Val: (92, 8)


In [4]:
drop_cols = ["window_id", "label"]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df["label"]

X_val = val_df.drop(columns=drop_cols)
y_val = val_df["label"]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)


X_train: (92, 6)
X_val: (92, 6)


In [5]:
n_pos = (y_train == 1).sum()
n_neg = (y_train == 0).sum()

scale_pos_weight = n_neg / n_pos

print("Positive:", n_pos)
print("Negative:", n_neg)
print("scale_pos_weight:", scale_pos_weight)


Positive: 46
Negative: 46
scale_pos_weight: 1.0


In [6]:
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42
)


In [7]:
model.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=True
)


[0]	validation_0-logloss:0.65100	validation_1-logloss:0.65100
[1]	validation_0-logloss:0.61064	validation_1-logloss:0.61064
[2]	validation_0-logloss:0.57529	validation_1-logloss:0.57529
[3]	validation_0-logloss:0.54079	validation_1-logloss:0.54079
[4]	validation_0-logloss:0.50916	validation_1-logloss:0.50916
[5]	validation_0-logloss:0.48077	validation_1-logloss:0.48077
[6]	validation_0-logloss:0.45375	validation_1-logloss:0.45375
[7]	validation_0-logloss:0.42890	validation_1-logloss:0.42890
[8]	validation_0-logloss:0.40555	validation_1-logloss:0.40555
[9]	validation_0-logloss:0.38361	validation_1-logloss:0.38361
[10]	validation_0-logloss:0.36303	validation_1-logloss:0.36303
[11]	validation_0-logloss:0.34409	validation_1-logloss:0.34409
[12]	validation_0-logloss:0.32728	validation_1-logloss:0.32728
[13]	validation_0-logloss:0.31058	validation_1-logloss:0.31058
[14]	validation_0-logloss:0.29478	validation_1-logloss:0.29478
[15]	validation_0-logloss:0.28001	validation_1-logloss:0.28001
[1

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [8]:
val_probs = model.predict_proba(X_val)[:, 1]
val_preds = (val_probs > 0.5).astype(int)

precision = precision_score(y_val, val_preds)
recall = recall_score(y_val, val_preds)
f1 = f1_score(y_val, val_preds)
roc_auc = roc_auc_score(y_val, val_probs)

print("Validation Results:")
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("ROC AUC:", roc_auc)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_preds))


Validation Results:
Precision: 1.0
Recall: 1.0
F1: 1.0
ROC AUC: 1.0

Confusion Matrix:
[[46  0]
 [ 0 46]]


In [9]:
thresholds = np.linspace(0.01, 0.99, 100)

best_f1 = 0
best_threshold = 0.5

for t in thresholds:
    preds = (val_probs > t).astype(int)
    f1_t = f1_score(y_val, preds)

    if f1_t > best_f1:
        best_f1 = f1_t
        best_threshold = t

print("Best threshold:", best_threshold)
print("Best F1:", best_f1)


Best threshold: 0.039696969696969696
Best F1: 1.0


In [10]:
importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)

importance_df


,feature,importance
1,pressure_drop,0.312086
3,mean,0.277947
5,range,0.252022
4,std,0.157946
0,slope,0.000000
2,min_position,0.000000


In [11]:
model.save_model(f"{WINDOW_DIR}/xgb_vortex_model.json")
print("Model saved.")


Model saved.
